# 04 · DermAugment: interactive demo
[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/shreyu20sharma/fastdiff-med/blob/main/notebooks/04_demo_gradio.ipynb)

A Gradio app (`app/app.py`) for a clinical ML team that needs extra training images of a rare lesion
type:
* **Generate:** choose a lesion class, a number of images and an engine (Baseline DDPM-1000,
  FastDiff DDIM-20 or Student 4-step), with a live timer. **Compare all engines** runs all three on
  the same request side by side.
* **Impact:** the augmentation results from notebook 03.

`share=True` prints a public `*.gradio.live` link (valid for 72 h) for recording the demo GIF.
For a permanent link, see "Hugging Face Space" at the end.

### Setup
Works on **Google Colab** (saves to `MyDrive/fastdiff-med/`) and on **Kaggle** (saves to
`/kaggle/working/fastdiff-med/` and picks up uploaded checkpoints from `/kaggle/input/`). It clones or
updates the repo and installs dependencies. Training and distillation resume from their last
checkpoint, so after a disconnect you only rerun this notebook.

* `QUICK = True`: ~5-minute smoke test with tiny settings (outputs go to a separate `quick/` folder).
* `LITE = True`: about half the GPU time (fewer FID samples, shorter distillation rounds). The results
  are still valid, just a bit less precise.

*Colab: Runtime → Change runtime type → **T4 GPU**. Kaggle: Settings → Accelerator → **GPU T4**, Internet **on**.*

In [ ]:
import os, sys, glob, shutil
QUICK = False   # True = tiny smoke-test settings (~5 min)
LITE = True     # True = about half the GPU time (recommended if GPU time is limited)

try:
    import google.colab  # noqa: F401
    IN_COLAB = True
except ImportError:
    IN_COLAB = False
IN_KAGGLE = os.path.exists('/kaggle/working')

if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
    BASE, CODE = '/content/drive/MyDrive/fastdiff-med', '/content/fastdiff-med'
elif IN_KAGGLE:
    BASE, CODE = '/kaggle/working/fastdiff-med', '/kaggle/working/fastdiff-med-code'
else:  # local Jupyter, started from the repo or its notebooks/ folder
    CODE = os.path.dirname(os.getcwd()) if os.path.basename(os.getcwd()) == 'notebooks' else os.getcwd()
    BASE = os.path.join(CODE, 'workdir')

if IN_COLAB or IN_KAGGLE:
    if not os.path.exists(CODE):
        !git clone -q https://github.com/shreyu20sharma/fastdiff-med.git {CODE}
    %cd {CODE}
    !git pull -q
    !pip install -q -r requirements.txt
else:
    %cd {CODE}
if CODE not in sys.path:
    sys.path.insert(0, CODE)

ROOT = BASE + ('/quick' if QUICK else '')
CKPT, RESULTS, DOCS = f'{ROOT}/checkpoints', f'{ROOT}/results', f'{ROOT}/docs'
os.environ['MEDMNIST_ROOT'] = f'{BASE}/data'  # dataset cache (~20 MB)
for d in (CKPT, RESULTS, DOCS, os.environ['MEDMNIST_ROOT']):
    os.makedirs(d, exist_ok=True)

if IN_KAGGLE:  # copy checkpoints from an uploaded Kaggle dataset, keeping the A/ B/ D/ layout
    for f in glob.glob('/kaggle/input/**/*', recursive=True):
        rel = f.split('checkpoints/', 1)[-1] if 'checkpoints/' in f else None
        if rel and os.path.isfile(f) and not os.path.exists(f'{CKPT}/{rel}'):
            os.makedirs(os.path.dirname(f'{CKPT}/{rel}'), exist_ok=True)
            shutil.copy(f, f'{CKPT}/{rel}')
    print('checkpoints:', sorted(glob.glob(f'{CKPT}/**/*.pt', recursive=True)))

import torch
print('GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'none (no GPU: only the light cells will work)')
print('Saving to', ROOT)

In [ ]:
from app.app import build_demo, load_engines
engines = load_engines(CKPT)          # Baseline = A/ema.pt, FastDiff = D/ema.pt, Student = distill/student_4step.pt
demo = build_demo(engines, results_dir=RESULTS)
demo.queue().launch(share=True, debug=False)

### Optional: permanent demo on Hugging Face Spaces (free CPU)
1. Create a Gradio Space at <https://huggingface.co/new-space>.
2. Upload `app/app.py` **as `app.py`** at the Space root, the `src/` folder and `requirements.txt`, and
   keep the same layout for checkpoints: `checkpoints/A/ema.pt`, `checkpoints/D/ema.pt`,
   `checkpoints/distill/student_4step.pt`. Optionally add `results/` for the Impact tab.
3. The 4-step student generates on the free CPU in about a second; the 1000-step baseline takes
   minutes there, which makes the speed difference obvious.

In [ ]:
# Collect the Space files into one folder on Drive, ready to upload
import shutil
SPACE = '/content/drive/MyDrive/fastdiff-med/hf_space'
shutil.rmtree(SPACE, ignore_errors=True)
shutil.copytree('src', f'{SPACE}/src', ignore=shutil.ignore_patterns('__pycache__'))
shutil.copy('app/app.py', f'{SPACE}/app.py')
shutil.copy('requirements.txt', f'{SPACE}/requirements.txt')
for rel in ['A/ema.pt', 'D/ema.pt', 'distill/student_4step.pt']:
    if os.path.exists(f'{CKPT}/{rel}'):
        os.makedirs(os.path.dirname(f'{SPACE}/checkpoints/{rel}'), exist_ok=True)
        shutil.copy(f'{CKPT}/{rel}', f'{SPACE}/checkpoints/{rel}')
if os.path.exists(RESULTS):
    shutil.copytree(RESULTS, f'{SPACE}/results', dirs_exist_ok=True)
!du -sh {SPACE}/*